# Train pipeline

## Setup envoironment

### Colab

In [ ]:
!git clone https://github.com/trxxnk/text-image-alignment.git

In [ ]:
import os
os.chdir("/content/text-image-alignment/")
print(f"Working directory: {os.getcwd()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!chmod +x src/scripts/setup_colab.sh
!src/scripts/setup_colab.sh

### Local

In [2]:
import os, sys
os.chdir(os.path.dirname(sys.prefix))
print(f"Working directory: {os.getcwd()}")

Working directory: /home/trxxnk/mycode/diplom


## Import libs

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
from torchvision.transforms import v2

import os
import json
import mlflow
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

In [4]:
from src.tps_dewarp.dataset import TPSDataset
from src.models.tpsresnet18 import TPSResNet18
from src.tps_dewarp.transforms import LetterboxResize

## Setup torch, dugshub, mlflow

In [5]:
SEED = 42
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [7]:
import dagshub
dagshub.init(repo_owner='trxxnk', repo_name='text-image-alignment', mlflow=True)

Initialized MLflow to track repo "trxxnk/text-image-alignment"

Repository trxxnk/text-image-alignment initialized!

## Dataset

### Load Dataset

In [8]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    LetterboxResize(256),
    v2.Normalize(mean=[0.5], std=[0.5]),
])

In [9]:
DATASET_DIR = "data/generated/v3"
dataset = TPSDataset(DATASET_DIR, transform=transform)
len(dataset)

33428

In [10]:
# Проверка
img, delta, difficulty = dataset[0]

print(img.shape)        # (1, 256, 256)
print(delta.shape)      # (N, 2)
print(difficulty)

torch.Size([1, 256, 256])
torch.Size([81, 2])
hard


In [11]:
# Проверка
from collections import Counter

diffs = [dataset.samples[i]["difficulty"] for i in range(len(dataset))]
counter = Counter(diffs)
total = counter.total()
probas = [val / total for val in counter.values()]
print(f"""
  {counter}
  {total=}
  probas={[round(val, 2) for val in probas]}
"""
)


  Counter({'easy': 13529, 'medium': 6763, 'hard': 6583, 'identity': 6553})
  total=33428
  probas=[0.2, 0.4, 0.2, 0.2]



### Train / Val / Test split

In [12]:
SEED = 42
N = len(dataset)

train_size = int(0.8 * N)
val_size = int(0.1 * N)
test_size = N - train_size - val_size

train_data, val_data, test_data = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

len(train_data), len(val_data), len(test_data)

(26742, 3342, 3344)

### Batching (DataLoaders)

In [34]:
NUM_WORKERS = 8
BATCH_SIZE = 64
PIN_MEMORY = True if device == torch.device("cuda") else False

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

In [14]:
x, y, d = next(iter(train_loader))

print(x.shape)  # (B, 1, 256, 256)
print(y.shape)  # (B, N, 2)
print(len(d))   # batch difficulties

torch.Size([128, 1, 256, 256])
torch.Size([128, 81, 2])
128


## Model

### Load Model

In [15]:
model = TPSResNet18(num_points=81, use_coordconv=True).to(device)

In [16]:
x = train_data[1][0]
x = x.unsqueeze(0).to(device)
out = model(x)
out.shape

torch.Size([1, 162])

In [17]:
train_data[1][1].shape

torch.Size([81, 2])

### Loss

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TPSLossRegularization(nn.Module):
    def __init__(self, grid_size=9, lambda_smooth=0.05):
        super().__init__()
        self.grid_size = grid_size
        self.lambda_smooth = lambda_smooth

    def forward(self, pred, target):
        B = pred.size(0)

        pred = pred.view(B, self.grid_size, self.grid_size, 2)
        target = target.view(B, self.grid_size, self.grid_size, 2)

        # ===== основной loss =====
        data_loss = F.smooth_l1_loss(pred, target)

        # ===== smoothness (очень важно для TPS) =====
        dx = pred[:, :, 1:, :] - pred[:, :, :-1, :]
        dy = pred[:, 1:, :, :] - pred[:, :-1, :, :]

        smooth_loss = dx.norm(dim=-1).mean() + dy.norm(dim=-1).mean()

        return data_loss + self.lambda_smooth * smooth_loss

In [19]:
loss_model = TPSLossRegularization(grid_size=9, lambda_smooth=0.05)

### Metrics

In [20]:
def compute_l2_px(pred, target, img_size=256):
    B = pred.shape[0]

    pred = pred.view(B, -1, 2)
    target = target.view(B, -1, 2)

    dist = torch.norm(pred - target, dim=2)  # (B, N)
    return dist.mean().item() * img_size

### Optimizer

In [21]:
opt = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

### Sheduler

In [22]:
EPOCHS = 40

In [23]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt,
    T_max=EPOCHS
)

## Train Loop

In [31]:
mlflow.set_experiment("TPS_Dewarp");
mlflow.start_run(run_name="test_02_local")

<ActiveRun: >

In [32]:
mlflow.log_params({
    "model": "resnet18 (no tanh)",
    "img_size": 256,
    "grid_size": 9,
    "batch_size": BATCH_SIZE,
    "lr": 1e-3,
    "optimizer": "AdamW",
    "scheduler": "CosineAnnealingLR",
    "loss": "TPSLossRegularization",
})

In [ ]:
CLIP_GRAD = 1.0

In [ ]:
try:
    best_val_loss = 1_000_000 

    for epoch in range(EPOCHS):

        # ===== TRAIN =====
        model.train()
        running_train_loss = 0.0

        train_loop = tqdm(train_loader, leave=False)

        for x, targets, _ in train_loop:

            x = x.to(device)
            targets = targets.to(device) 

            pred = model(x)
            loss = loss_model(pred, targets)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
            opt.step()

            running_train_loss += loss.item()

            train_loop.set_postfix({
                "loss": f"{loss.item():.4f}"
            })

        train_loss = running_train_loss / len(train_loader)

        # ===== VALID =====
        model.eval()
        running_val_loss = 0.0
        running_l2 = 0.0

        with torch.no_grad():
            for x, targets, _ in val_loader:

                x = x.to(device)
                targets = targets.to(device) 

                pred = model(x)

                loss = loss_model(pred, targets)
                running_val_loss += loss.item()

                l2 = compute_l2_px(pred, targets)
                running_l2 += l2

        val_loss = running_val_loss / len(val_loader)
        val_l2 = running_l2 / len(val_loader)

        scheduler.step()
        current_lr = opt.param_groups[0]["lr"]

        print(
            f"Epoch {epoch+1}/{EPOCHS} | "
            f"train={train_loss:.4f} | val={val_loss:.4f} | "
            f"L2={val_l2:.2f} | lr={current_lr:.2e}"
        )

        # ===== MLflow логирование =====
        mlflow.log_metrics({
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_l2_px": val_l2,
            "lr": current_lr
        }, step=epoch)

        # ===== save best =====
        if val_loss < best_val_loss:
            best_val_loss = val_loss

            # Сохраняем модель во временный файл и логируем в MLflow
            import tempfile
            from datetime import datetime
            # Формируем имя файла с эпохой, датой и временем
            dt_string = datetime.now().strftime("%Y%m%d_%H%M%S")
            model_filename = f"model_epoch{epoch+1}_{dt_string}.pth"
            with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as tmp:
                torch.save(model.state_dict(), tmp.name)
                # Сохраняем в MLflow c правильным именем в artifacts
                mlflow.log_artifact(tmp.name, artifact_path=f"models/{model_filename}")

finally:
    mlflow.end_run()

 20%|█▉        | 82/418 [09:38<38:32,  6.88s/it, loss=0.0009]  